# Dimension-conditioned ModernBERT regressor (plan steps 2 + 3)

Implements `plan.md` step 2 (train the conditioned model) and step 3 (the gate: swap test +
first scoring of the simulated set). Design per `plan.md` section 3-4:

- Input: `Dimension: <name>` + `Definition: <paragraph from dimensions.md>` + conversation.
  No sub-dimension line.
- Target: `effectiveness_consensus` (half-point floats 1.0-5.0), MSE regression.
- Backbone: `answerdotai/ModernBERT-large`, mean pooling, dropout 0.1 before the head.
- **GPU requirement (hard): Ampere or newer (A100 / L4), bf16.** ModernBERT was pretrained
  in bf16 and has known fp16 NaN issues; the T4 has no bf16 support. A guard cell below
  refuses to run on unsupported GPUs.
- Comparison bridge: our Stage-1 Longformer run (r=0.672, RMSE=0.815, MAE=0.607), not the
  paper's r=0.736 directly (different backbone, no sub-dim line, added definitions).

Requires `transformers>=4.48.0` (first release with native ModernBERT support).

In [1]:
!pip install "transformers>=4.48.0" datasets==2.19.0 accelerate evaluate scipy scikit-learn --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 19.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [2]:
import torch

assert torch.cuda.is_available(), "No GPU. Use an A100 or L4 runtime."
assert torch.cuda.is_bf16_supported(), (
    "This GPU does not support bf16 (you are probably on a T4). "
    "ModernBERT must not be trained in fp16 (known NaN issues) - switch the runtime to A100 or L4."
)
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-80GB


## Google Drive

Mounted once, up front, so a mount problem surfaces *before* training rather than after.
Expected layout under `MyDrive/PhD/AITutorEval/`:

- `data/conversations.jsonl` — upload `runs/convolearn/run_set_07081515/conversations.jsonl`
  there before running (needed by the gate at the end).
- `models/` — the trained model is copied here.

In [3]:
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

DRIVE_BASE = "/content/drive/MyDrive/PhD/AITutorEval"
DRIVE_MODELS = f"{DRIVE_BASE}/models"
SIM_PATH = f"{DRIVE_BASE}/data/conversations.jsonl"
os.makedirs(DRIVE_MODELS, exist_ok=True)

# Fail fast: the gate at the end needs the simulated set, so check it now, not after training.
assert os.path.exists(SIM_PATH), (
    f"Missing {SIM_PATH} - upload runs/convolearn/run_set_07081515/conversations.jsonl "
    "from the repo to MyDrive/PhD/AITutorEval/data/ and re-run this cell."
)
n_sim = sum(1 for _ in open(SIM_PATH, encoding="utf-8"))
print(f"Found simulated set: {SIM_PATH} ({n_sim} conversations)")

Mounted at /content/drive
Found simulated set: /content/drive/MyDrive/PhD/AITutorEval/data/conversations.jsonl (60 conversations)


In [4]:
from datasets import load_dataset

dataset = load_dataset("masharma/convolearn")
raw = dataset["train"]
print(raw)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Generating train split:   0%|          | 0/2134 [00:00<?, ? examples/s]

Dataset({
    features: ['kb_subdim', 'kb_dim', 'effectiveness_consensus', 'completeness_consensus', 'cleaned_conversation', 'earthscience_topic', 'num_exchanges'],
    num_rows: 2134
})


## Dimension definitions

Copied verbatim from `evaluations/pedagogy/dimensions.md` (the notebook runs on Colab with no
repo access, so the text is embedded). If `dimensions.md` changes, re-copy it here.

In [5]:
DIMENSIONS = {
    "Cognitive Engagement": (
        "Cognitive Engagement refers to the depth of processing and quality of thinking "
        "strategies students employ during learning (Blumenfeld et al., 2006; Chi & Wylie, 2014). "
        "In dialogic tutoring, cognitive engagement is the most direct behavioral counterpart to "
        "answer-giving: where an answer-giving tutor resolves cognitive challenge by providing "
        "solutions, a dialogically engaging tutor uses that challenge as the site of learning. "
        "Linguistically, it manifests as open-ended questioning, uptake of student ideas, and "
        "scaffolded elaboration rather than declarative explanation."
    ),
    "Formative Assessment": (
        "Formative Assessment refers to the ongoing, interactive monitoring of student "
        "understanding during instruction to regulate learning in real time (Cowie & Bell, 1999; "
        "Black & Wiliam, 2009). Unlike summative evaluation, it is embedded within the dialogue: "
        "tutors attend to student contributions, interpret them against learning goals, and adapt "
        "their next move accordingly. Linguistically, it appears as comprehension checks, probing "
        "follow-up questions, and responses that build on or correct student ideas."
    ),
    "Accountability": (
        "Accountability reflects expectations that discourse aligns with norms of evidence and "
        "reasoning (Michaels et al., 2008). In dialogic tutoring, accountability moves the "
        "conversation beyond mere exchange of opinions toward epistemic responsibility: students "
        "are expected to justify claims, evaluate evidence, and engage with counterarguments. "
        "Linguistically, it manifests as tutor prompts that require students to cite evidence, "
        "explain their reasoning, or defend a position."
    ),
    "Cultural Responsiveness": (
        "Cultural Responsiveness recognizes that effective instruction must engage learners' "
        "diverse cultural backgrounds and build on their prior knowledge and experiences "
        "(Ladson-Billings, 1995; Au et al., 1981). In dialogic tutoring, it requires tutors to "
        "connect concepts to culturally relevant contexts and affirm diverse ways of knowing "
        "rather than assuming a single canonical frame. Linguistically, it manifests as analogies "
        "drawn from students' cultural contexts and invitations to connect content to their "
        "experience."
    ),
    "Metacognition": (
        "Metacognition refers to awareness and regulation of one's own cognitive processes "
        "(Flavell, 1981). In dialogic tutoring, it involves prompting reflection and modeling "
        "through thinking aloud, making expert reasoning visible and encouraging students to "
        "adopt similar habits. Linguistically, it appears as prompts asking students to explain "
        "their reasoning, identify difficulties, or compare current understanding with prior "
        "beliefs."
    ),
    "Power Dynamics": (
        "Power Dynamics captures how agency and participation are distributed in learning "
        "interactions (Gordon & Foucault, 1980). In dialogic tutoring, equitable power dynamics "
        "are essential: tutors who dominate the conversational floor or position themselves as "
        "sole arbiters of knowledge undermine the collaborative construction of understanding. "
        "Linguistically, it manifests in turn-taking patterns, the degree to which student ideas "
        "are taken up and built upon, and whether students are positioned as active contributors."
    ),
}

ALL_DIMS = sorted(DIMENSIONS)
assert set(ALL_DIMS) == set(raw.unique("kb_dim")), (
    "DIMENSIONS keys do not match the dataset's kb_dim values: "
    f"{set(ALL_DIMS) ^ set(raw.unique('kb_dim'))}"
)
print("Dimensions:", ALL_DIMS)

Dimensions: ['Accountability', 'Cognitive Engagement', 'Cultural Responsiveness', 'Formative Assessment', 'Metacognition', 'Power Dynamics']


In [6]:
from datasets import DatasetDict
import numpy as np

# Same seeded topic-stratified 70/15/15 split as the other notebooks, for comparability.
topic_to_indices = {}
for i, topic in enumerate(raw["earthscience_topic"]):
    topic_to_indices.setdefault(topic, []).append(i)

train_indices, val_indices, test_indices = [], [], []
rng = np.random.default_rng(seed=42)

for topic, indices in topic_to_indices.items():
    indices = np.array(indices)
    rng.shuffle(indices)
    n = len(indices)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_indices.extend(indices[:n_train].tolist())
    val_indices.extend(indices[n_train:n_train + n_val].tolist())
    test_indices.extend(indices[n_train + n_val:].tolist())

dataset_dict = DatasetDict({
    "train": raw.select(train_indices),
    "validation": raw.select(val_indices),
    "test": raw.select(test_indices),
})
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['kb_subdim', 'kb_dim', 'effectiveness_consensus', 'completeness_consensus', 'cleaned_conversation', 'earthscience_topic', 'num_exchanges'],
        num_rows: 1491
    })
    validation: Dataset({
        features: ['kb_subdim', 'kb_dim', 'effectiveness_consensus', 'completeness_consensus', 'cleaned_conversation', 'earthscience_topic', 'num_exchanges'],
        num_rows: 318
    })
    test: Dataset({
        features: ['kb_subdim', 'kb_dim', 'effectiveness_consensus', 'completeness_consensus', 'cleaned_conversation', 'earthscience_topic', 'num_exchanges'],
        num_rows: 325
    })
})


In [7]:
def format_input(conversation_text, dim_name):
    """Conditioning block + conversation. Also used at inference / in the gate cells,
    where dim_name is asserted rather than taken from the row."""
    return (
        f"Dimension: {dim_name}\n"
        f"Definition: {DIMENSIONS[dim_name]}\n\n"
        f"Conversation:\n{conversation_text}"
    )

dataset_dict = dataset_dict.map(
    lambda ex: {"input_text": format_input(ex["cleaned_conversation"], ex["kb_dim"])}
)
print(dataset_dict["train"][0]["input_text"][:600])

Map:   0%|          | 0/1491 [00:00<?, ? examples/s]

Map:   0%|          | 0/318 [00:00<?, ? examples/s]

Map:   0%|          | 0/325 [00:00<?, ? examples/s]

Dimension: Metacognition
Definition: Metacognition refers to awareness and regulation of one's own cognitive processes (Flavell, 1981). In dialogic tutoring, it involves prompting reflection and modeling through thinking aloud, making expert reasoning visible and encouraging students to adopt similar habits. Linguistically, it appears as prompts asking students to explain their reasoning, identify difficulties, or compare current understanding with prior beliefs.

Conversation:
Student: During photosynthesis, what gas do plants take from the atmosphere and how does this connect to the carbon c


In [8]:
from transformers import AutoTokenizer

BASE_MODEL = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Pick max_length from the 99th percentile of *formatted* training inputs (the definition
# adds ~120-150 tokens on top of the conversation), rounded up to a multiple of 64.
# Truncation cuts the conversation tail; the conditioning block at the front is never lost.
train_lengths = np.array([
    len(tokenizer(t, truncation=False)["input_ids"]) for t in dataset_dict["train"]["input_text"]
])
p99 = int(np.percentile(train_lengths, 99))
MAX_LENGTH = min(int(np.ceil(p99 / 64)) * 64, 4096)
print(f"formatted tokens - mean: {train_lengths.mean():.0f}, median: {np.median(train_lengths):.0f}, "
      f"p99: {p99}, max: {train_lengths.max()}")
print(f"MAX_LENGTH = {MAX_LENGTH} "
      f"(truncates {(train_lengths > MAX_LENGTH).mean():.2%} of train examples)")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (13804 > 8192). Running this sequence through the model will result in indexing errors


formatted tokens - mean: 826, median: 752, p99: 1965, max: 13804
MAX_LENGTH = 1984 (truncates 1.01% of train examples)


In [9]:
from transformers import DataCollatorWithPadding

def tokenize(example):
    encoding = tokenizer(example["input_text"], max_length=MAX_LENGTH, truncation=True, padding=False)
    encoding["labels"] = float(example["effectiveness_consensus"])
    return encoding

tokenized = dataset_dict.map(tokenize, remove_columns=dataset_dict["train"].column_names)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(tokenized)

Map:   0%|          | 0/1491 [00:00<?, ? examples/s]

Map:   0%|          | 0/318 [00:00<?, ? examples/s]

Map:   0%|          | 0/325 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1491
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 318
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 325
    })
})


In [10]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from scipy.stats import pearsonr, spearmanr

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=1,
    problem_type="regression",
    classifier_pooling="mean",   # attention-masked mean pooling (ModernBERT-native option)
    classifier_dropout=0.1,      # dropout before the regression head (plan section 4)
    attn_implementation="sdpa",  # no flash-attn dependency
)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.squeeze(preds)
    return {
        "rmse": float(np.sqrt(np.mean((preds - labels) ** 2))),
        "mae": float(np.mean(np.abs(preds - labels))),
        "pearson": float(pearsonr(preds, labels)[0]),
        "spearman": float(spearmanr(preds, labels)[0]),
    }

# Batch 8 x accum 2 = effective 16. Fits A100-40GB at this seq length; on L4 (24GB)
# drop per_device_train_batch_size to 4 and double gradient_accumulation_steps.
training_args = TrainingArguments(
    output_dir="./modernbert-conditioned",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    bf16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Rmse,Mae,Pearson,Spearman
1,6.679297,0.644739,0.802957,0.629963,0.712440,0.728366
2,1.363449,0.632454,0.795269,0.605985,0.696253,0.705349
3,0.975355,0.573369,0.757212,0.606476,0.715473,0.731072
4,0.654710,0.702564,0.838191,0.621855,0.726106,0.744446
5,0.300618,0.518517,0.720081,0.547637,0.733079,0.744899


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=470, training_loss=1.519306300548797, metrics={'train_runtime': 373.7159, 'train_samples_per_second': 19.948, 'train_steps_per_second': 1.258, 'total_flos': 1.920080376873947e+16, 'train_loss': 1.519306300548797, 'epoch': 5.0})

## Test-set evaluation (overall + per-dimension)

Per-dimension numbers are what actually matter for the goal (plan section 7); the overall row
is the continuity bridge to Stage 1 (Longformer: r=0.672, RMSE=0.815, MAE=0.607).

In [11]:
test_out = trainer.predict(tokenized["test"])
test_preds = np.squeeze(test_out.predictions)
test_labels = np.array(test_out.label_ids)
test_dims = np.array(dataset_dict["test"]["kb_dim"])

def metric_row(preds, labels):
    return {
        "n": len(labels),
        "pearson": pearsonr(preds, labels)[0] if len(labels) > 1 else float("nan"),
        "spearman": spearmanr(preds, labels)[0] if len(labels) > 1 else float("nan"),
        "rmse": float(np.sqrt(np.mean((preds - labels) ** 2))),
        "mae": float(np.mean(np.abs(preds - labels))),
    }

rows = {"OVERALL": metric_row(test_preds, test_labels)}
for dim in ALL_DIMS:
    mask = test_dims == dim
    rows[dim] = metric_row(test_preds[mask], test_labels[mask])

print(f"{'':<26}{'n':>5} {'pearson':>9} {'spearman':>9} {'rmse':>7} {'mae':>7}")
for name, r in rows.items():
    print(f"{name:<26}{r['n']:>5} {r['pearson']:>9.3f} {r['spearman']:>9.3f} {r['rmse']:>7.3f} {r['mae']:>7.3f}")
print("\nStage-1 Longformer bridge: pearson 0.672, rmse 0.815, mae 0.607")

                              n   pearson  spearman    rmse     mae
OVERALL                     325     0.702     0.710   0.767   0.578
Accountability               45     0.749     0.787   0.679   0.582
Cognitive Engagement         83     0.688     0.681   0.703   0.507
Cultural Responsiveness      24     0.178     0.217   1.139   0.870
Formative Assessment         36     0.816     0.752   0.610   0.510
Metacognition                87     0.702     0.684   0.726   0.564
Power Dynamics               50     0.711     0.750   0.884   0.627

Stage-1 Longformer bridge: pearson 0.672, rmse 0.815, mae 0.607


In [12]:
import json, shutil

model_path = "./models/conditioned_modernbert"
os.makedirs(model_path, exist_ok=True)
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
with open(f"{model_path}/run_config.json", "w") as f:
    json.dump({"base_model": BASE_MODEL, "max_length": MAX_LENGTH, "dims": ALL_DIMS}, f, indent=2)
print(f"Saved to {model_path}")

shutil.copytree(model_path, f"{DRIVE_MODELS}/conditioned_modernbert", dirs_exist_ok=True)
print(f"Copied to Google Drive: {DRIVE_MODELS}/conditioned_modernbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to ./models/conditioned_modernbert
Copied to Google Drive: /content/drive/MyDrive/PhD/AITutorEval/models/conditioned_modernbert


### Reloading without retraining

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import json

model_path = "/content/drive/MyDrive/PhD/AITutorEval/models/conditioned_modernbert"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path).cuda().eval()
MAX_LENGTH = json.load(open(f"{model_path}/run_config.json"))["max_length"]
```

Trainer-native model class, so no custom module definition is needed. The `DIMENSIONS` dict
and `format_input` cells must still be run first - inference inputs are built with them.

## Gate (plan step 3): does the model actually use the dimension?

Two checks, run against the just-trained checkpoint. Both must pass before any ablation work;
if either fails, the multi-head contingency (plan section 6) fires.

1. **Swap test** (unlabeled): score each test conversation under all 6 dimensions. If scores
   barely move when the conditioning changes, the model ignores the dimension text.
2. **Simulated set** (labeled) - **first of exactly two touches** of this set (plan section 8a).
   Conversations engineered vh/vl on one dimension: the vh-vl gap should be large on the
   targeted dimension and concentrated there.

In [13]:
def score_conversations(conv_texts, dim_name, batch_size=32):
    """Score raw conversation texts on an asserted dimension. Returns np.array of floats."""
    m = trainer.model.eval()
    device = next(m.parameters()).device
    scores = []
    with torch.no_grad():
        for i in range(0, len(conv_texts), batch_size):
            batch = [format_input(t, dim_name) for t in conv_texts[i:i + batch_size]]
            enc = tokenizer(batch, max_length=MAX_LENGTH, truncation=True,
                            padding=True, return_tensors="pt").to(device)
            out = m(**enc)
            scores.extend(out.logits.squeeze(-1).float().cpu().tolist())
    return np.array(scores)

In [14]:
# --- Check 1: swap test on the test split ---
test_convs = dataset_dict["test"]["cleaned_conversation"]
true_dims = np.array(dataset_dict["test"]["kb_dim"])
labels = np.array(dataset_dict["test"]["effectiveness_consensus"], dtype=float)

score_matrix = np.stack([score_conversations(test_convs, d) for d in ALL_DIMS], axis=1)  # (N, 6)
dim_col = {d: i for i, d in enumerate(ALL_DIMS)}
true_idx = np.array([dim_col[d] for d in true_dims])
true_scores = score_matrix[np.arange(len(labels)), true_idx]

spread = score_matrix.max(axis=1) - score_matrix.min(axis=1)
print(f"Per-conversation score spread across the 6 dimensions "
      f"(on a 1-5 scale): mean {spread.mean():.3f}, median {np.median(spread):.3f}")

r_true = pearsonr(true_scores, labels)[0]
r_swapped = []
for j, d in enumerate(ALL_DIMS):
    mask = true_idx != j  # conversations for which dimension d is a *wrong* label
    r_swapped.append(pearsonr(score_matrix[mask, j], labels[mask])[0])
print(f"pearson(true-dimension score, label):        {r_true:.3f}")
print(f"pearson(swapped-dimension score, label):     mean {np.mean(r_swapped):.3f} "
      f"(per dim: {', '.join(f'{d.split()[0]} {r:.3f}' for d, r in zip(ALL_DIMS, r_swapped))})")
print("\nFAIL signs: spread near 0, or swapped correlations ~= the true-dimension correlation")
print("(both mean the model scores generic quality and ignores the conditioning).")

Per-conversation score spread across the 6 dimensions (on a 1-5 scale): mean 0.483, median 0.469
pearson(true-dimension score, label):        0.702
pearson(swapped-dimension score, label):     mean 0.670 (per dim: Accountability 0.663, Cognitive 0.664, Cultural 0.699, Formative 0.652, Metacognition 0.677, Power 0.664)

FAIL signs: spread near 0, or swapped correlations ~= the true-dimension correlation
(both mean the model scores generic quality and ignores the conditioning).


In [15]:
# --- Check 2: simulated set (touch 1 of 2) ---
# SIM_PATH was set and existence-checked in the Drive cell at the top of the notebook.
from sklearn.metrics import roc_auc_score

LABEL2DIM = {"ac": "Accountability", "ce": "Cognitive Engagement", "fa": "Formative Assessment"}

sim = [json.loads(line) for line in open(SIM_PATH, encoding="utf-8")]
sim_texts = [r["text"] for r in sim]
sim_target = np.array([LABEL2DIM[r["label"].split("-")[0]] for r in sim])
sim_level = np.array([r["label"].split("-")[1] for r in sim])  # vh / vl
print(f"{len(sim)} conversations, targets: {sorted(set(sim_target))}")

sim_matrix = np.stack([score_conversations(sim_texts, d) for d in ALL_DIMS], axis=1)  # (60, 6)

print(f"\nvh-vl mean gap by (engineered dimension x scored dimension); "
      f"** = the targeted dimension, where the gap should be largest:")
header = " " * 24 + "".join(f"{d.split()[0][:10]:>12}" for d in ALL_DIMS)
print(header)
for tgt in sorted(set(sim_target)):
    row_mask = sim_target == tgt
    vh, vl = row_mask & (sim_level == "vh"), row_mask & (sim_level == "vl")
    gaps = sim_matrix[vh].mean(axis=0) - sim_matrix[vl].mean(axis=0)
    cells = "".join(
        f"{'**' if ALL_DIMS[j] == tgt else '  '}{gaps[j]:>8.2f}{'**' if ALL_DIMS[j] == tgt else '  '}"
        for j in range(len(ALL_DIMS))
    )
    print(f"{tgt:<24}{cells}")

print("\nvh-vs-vl AUC on the targeted dimension's score (1.0 = perfect separation):")
for tgt in sorted(set(sim_target)):
    row_mask = sim_target == tgt
    y = (sim_level[row_mask] == "vh").astype(int)
    auc = roc_auc_score(y, sim_matrix[row_mask, dim_col[tgt]])
    print(f"  {tgt:<24}{auc:.3f}")

60 conversations, targets: [np.str_('Accountability'), np.str_('Cognitive Engagement'), np.str_('Formative Assessment')]

vh-vl mean gap by (engineered dimension x scored dimension); ** = the targeted dimension, where the gap should be largest:
                          Accountabi   Cognitive    Cultural   Formative  Metacognit       Power
Accountability          **    0.46**      0.39        0.52        0.44        0.50        0.50  
Cognitive Engagement          0.42  **    0.42**      0.49        0.44        0.47        0.48  
Formative Assessment          0.53        0.51        0.55  **    0.53**      0.57        0.55  

vh-vs-vl AUC on the targeted dimension's score (1.0 = perfect separation):
  Accountability          0.960
  Cognitive Engagement    0.995
  Formative Assessment    1.000


### Interpreting the gate (plan sections 5, 6, 8a)

**Pass** = all of:
- Swap test: per-conversation spread is clearly nonzero and the true-dimension correlation
  beats the swapped-dimension correlations.
- Simulated set: vh-vl gap is positive and **largest on the targeted dimension** for all three
  engineered dimensions (an ac-vh/ac-vl contrast equally visible on every column = generic
  quality, not the construct), with high AUC on the targeted dimension.

**Any failure** -> the multi-head contingency fires (plan section 6): train
`ModernBertMultiHead.ipynb` (switch its `fp16=True` to `bf16=True` first).

**On pass** -> proceed to ablations (plan step 5), making all decisions on the validation
split. The simulated set is now spent until its second and final touch at step 7.